# NYSE Glitch Exercise: A Python/Pandas Investigation

- See article: https://www.reuters.com/markets/us/some-nyse-listed-stocks-briefly-halted-trading-after-market-open-2023-01-24/
- Exchange spreadsheet: 	https://www.nyse.com/publicdocs/nyse/notifications/market-status/110000531402/Final_NYSE_Auctions.xlsx
- Yahoo Finance: https://finance.yahoo.com/quote/UBER/history/

---

## The Morning Everything Went Wrong

**January 24, 2023, 9:30 AM EST**

The opening bell rang at the New York Stock Exchange, but something was terribly wrong. Over 250 stocks—including blue chips like ExxonMobil, McDonald's, and Uber—failed to execute their opening auctions. Prices went haywire. Trades that should never have happened... happened.

By the end of the day, the NYSE would bust hundreds of trades, the SEC would launch an investigation, and somewhere, in a nondescript office building, a small team of quantitative traders would be quietly celebrating.

---

## The Vultures Who Wait

Here's what most people don't know about market microstructure: **glitches are predictable in their unpredictability.**

Every major exchange has roughly one significant technical failure per year. It's not a matter of *if*, but *when*. And a small subset of hedge funds have built entire strategies around this simple fact.

These aren't your typical high-frequency traders racing to shave microseconds. These are **"chaos arbitrageurs"**—patient predators who maintain dormant systems year-round, constantly calculating one thing: *the bust bands*.

### The Bust Band Strategy

When an exchange experiences a technical failure, it doesn't simply let erroneous trades stand. Instead, it defines **bust bands**—price ranges outside of which trades will be declared null and void. The key insight? These bands follow predictable rules based on previous closing prices, reference prices, and exchange-specific rulesets.

A sophisticated fund can calculate these bands in real-time. When chaos strikes, they know exactly how far prices can deviate before trades get cancelled. They position themselves at the **edge of the abyss**—the maximum allowable price deviation—and wait.

---

## Your Mission

You are a forensic market analyst investigating whether chaos arbitrage occurred in UBER on January 24, 2023.

### Data Sources

| Source | Description |
|--------|-------------|
| NYSE Official Bust List | `https://www.nyse.com/publicdocs/nyse/notifications/market-status/110000531402/Final_NYSE_Auctions.xlsx` |
| Historical Prices | Yahoo Finance via `yfinance` |
| Tick Data | Trade fills provided below |

### Tick Data from That Morning

```python
fillstring = '''
TU|UBER:*|1465|20230124 09:30:00.174729189|25.67|12|N|h50|PREOPEN|236968|C|h08|32582
TU|UBER:*|1466|20230124 09:30:00.174734304|25.71|6206|N|h77|REGULAR|243174|C|h0A|127
TU|UBER:*|1467|20230124 09:30:00.174739474|25.79|55|N|h50|REGULAR|243229|C|h0A|32582
TU|UBER:*|1468|20230124 09:30:00.174744669|25.84|1545|N|h72|REGULAR|244774|C|h05|127
TU|UBER:*|1469|20230124 09:30:00.174750169|25.87|422|N|h72|REGULAR|245196|C|h05|127
TU|UBER:*|1470|20230124 09:30:00.174750169|28.51|4575|N|h72|REGULAR|249771|C|h05|127
'''
```

Column 4 is the fill price. Column 5 is the number of shares.

---

## Deliverables

**Using pandas, answer the following:**

1. **What were the bust bands for UBER?** Extract `Trades_Busted_Below_This_Price` and `Trades_Busted_Above_This_Price` from the NYSE spreadsheet. Note: trades *at* the boundary price are busted, so valid trades must be strictly inside.

2. **Identify which trades from the tick data were busted vs. survived.** Show your filtering logic.

3. **Did chaos arbitrage occur?** Examine the surviving trade(s) closest to the bust boundary. What is the dollar profit/loss if held until that day's close? Use `yfinance` to get UBER's closing price.


In [23]:
import yfinance as yf


In [24]:
# cache results so we don't keep hitting the API
from pathlib import Path
import pandas as pd
import io

In [25]:
# set current directory and cache file path
current_dir = Path.cwd()
cache_file = current_dir / 'uber_data.pkl'

In [26]:
if cache_file.exists():
    # read data from cache
    df = pd.read_pickle(cache_file)
else:
    df = yf.download(['UBER'],period='10y')
    df.to_pickle(cache_file)

In [27]:
%ls

Final_NYSE_Auctions.xlsx
NYSE_Glitch_Exercise_Python_Pandas_Investigation.ipynb
NYSE_Glitch_Reuters-Article.pdf
README.md
uber_data.pkl


In [28]:
spread_sheet = pd.read_excel('Final_NYSE_Auctions.xlsx')

In [29]:
spread_sheet.head()

,SYMB,LULD_Update_Received_Time,Trades_Busted_Below_This_Price,Trades_Busted_Above_This_Price
0,ABC,2023-01-24 09:31:00.010675456,153.63400,169.80600
1,ABT,2023-01-24 09:30:00.105743104,108.11950,119.50050
2,ADM,2023-01-24 09:30:00.108570368,79.80000,88.20000
3,AI,2023-01-24 09:30:00.189420544,13.00491,15.89489
4,ALB,2023-01-24 09:30:00.124898816,244.15000,269.85000


In [30]:
#set index to SYMB
spread_sheet = spread_sheet.set_index('SYMB')

In [31]:
uber_bustband = spread_sheet.loc['UBER']

In [32]:
uber_bustband

LULD_Update_Received_Time         2023-01-24 09:30:00.175080192
Trades_Busted_Below_This_Price                             28.5
Trades_Busted_Above_This_Price                             31.5
Name: UBER, dtype: object

In [33]:
uber_jan_24_23 = df.loc['2023-01-24']

In [34]:
fillstring = '''
TU|UBER:*|1465|20230124 09:30:00.174729189|25.67|12|N|h50|PREOPEN|236968|C|h08|32582
TU|UBER:*|1466|20230124 09:30:00.174734304|25.71|6206|N|h77|REGULAR|243174|C|h0A|127
TU|UBER:*|1467|20230124 09:30:00.174739474|25.79|55|N|h50|REGULAR|243229|C|h0A|32582
TU|UBER:*|1468|20230124 09:30:00.174744669|25.84|1545|N|h72|REGULAR|244774|C|h05|127
TU|UBER:*|1469|20230124 09:30:00.174750169|25.87|422|N|h72|REGULAR|245196|C|h05|127
TU|UBER:*|1470|20230124 09:30:00.174750169|28.51|4575|N|h72|REGULAR|249771|C|h05|127
'''

In [35]:
split_fillstring = fillstring.strip().split('\n')

In [36]:
columns = [
    'msg_type', 'symbol', 'seq_id', 'timestamp', 
    'price', 'quantity', 'flag', 'h_code1', 
    'session', 'trade_id', 'status', 'h_code2', 'ref_id'
]

In [37]:
# Read the string directly using the pipe delimiter
trades = pd.read_csv(io.StringIO(fillstring), sep='|', names=columns)
trades['price'] = pd.to_numeric(trades['price'])
trades['quantity'] = pd.to_numeric(trades['quantity'])

print(trades[['timestamp', 'price', 'quantity']])

                     timestamp  price  quantity
0  20230124 09:30:00.174729189  25.67        12
1  20230124 09:30:00.174734304  25.71      6206
2  20230124 09:30:00.174739474  25.79        55
3  20230124 09:30:00.174744669  25.84      1545
4  20230124 09:30:00.174750169  25.87       422
5  20230124 09:30:00.174750169  28.51      4575


In [38]:
# bust logic
trades['status'] = 'survived'
trades.loc[(trades['price'] < uber_bustband.Trades_Busted_Below_This_Price) | 
           (trades['price'] > uber_bustband.Trades_Busted_Above_This_Price), 'status'] = 'busted'

In [39]:
trades

,msg_type,symbol,seq_id,timestamp,price,quantity,flag,h_code1,session,trade_id,status,h_code2,ref_id
0,TU,UBER:*,1465,20230124 09:30:00.174729189,25.67,12,N,h50,PREOPEN,236968,busted,h08,32582
1,TU,UBER:*,1466,20230124 09:30:00.174734304,25.71,6206,N,h77,REGULAR,243174,busted,h0A,127
2,TU,UBER:*,1467,20230124 09:30:00.174739474,25.79,55,N,h50,REGULAR,243229,busted,h0A,32582
3,TU,UBER:*,1468,20230124 09:30:00.174744669,25.84,1545,N,h72,REGULAR,244774,busted,h05,127
4,TU,UBER:*,1469,20230124 09:30:00.174750169,25.87,422,N,h72,REGULAR,245196,busted,h05,127
5,TU,UBER:*,1470,20230124 09:30:00.174750169,28.51,4575,N,h72,REGULAR,249771,survived,h05,127


In [41]:
# 3. **Did chaos arbitrage occur?** Examine the surviving trade(s) closest to the bust boundary. What is the dollar profit/loss if held until that day's close? Use `yfinance` to get UBER's closing price.
uber_close = uber_jan_24_23[('Close', 'UBER')]

Profit=(Close Price−Execution Price)×Quantity

In [45]:
surviving_trades = trades[trades['status'] == 'survived'].copy()

# Calculate P/L based on the close price
surviving_trades['pl'] = (uber_close - surviving_trades['price']) * surviving_trades['quantity']


total_chaos_profit = surviving_trades['pl'].sum()

print(f"Total Chaos Arbitrage Profit: ${total_chaos_profit:,.2f}")

Total Chaos Arbitrage Profit: $6,496.50
